# Power Spectral Density (PSD)

**Dataset**: PhysioNet Auditory EEG  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

PSD reveals how signal energy is distributed across frequencies. We use Welch's method to compute PSD for all 4 channels and analyze energy distribution across the five brain wave bands.

## Expected outputs

- Spectrum plot for all 4 channels on log scale
- Bar chart of band power distribution for channel P4
- Energy concentrated in low bands, decreasing in high bands

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| nperseg | 1024 | Segment length |
| noverlap | 512 | Overlap between segments |
| Bands | 5 | Delta, Theta, Alpha, Beta, Gamma |


## 1. Install dependencies


In [ ]:
!pip install scipy numpy plotly mne wfdb


## 2. Clone repo and download data

We download only subject 1 (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2.


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


## 4. Apply PSD analysis

We compute PSD using Welch's method for all channels, then analyze band power distribution.


In [ ]:
from scipy.signal import welch

NPERSEG = 1024
NOVERLAP = 512
BANDS = [
    ('Delta', 0.5, 4, 'green'),
    ('Theta', 4, 8, 'blue'),
    ('Alpha', 8, 13, 'orange'),
    ('Beta', 13, 30, 'red'),
    ('Gamma', 30, 80, 'purple'),
]

psd_results = {}
for i, name in enumerate(ch_names):
    freqs, psd = welch(eeg_data[:, i], fs=fs, nperseg=NPERSEG, noverlap=NOVERLAP)
    psd_results[name] = (freqs, psd)

freqs_p4, psd_p4 = psd_results[ch_names[0]]
band_powers = []
for name, fmin, fmax, color in BANDS:
    mask = (freqs_p4 >= fmin) & (freqs_p4 <= fmax)
    band_powers.append(np.trapezoid(psd_p4[mask], freqs_p4[mask]))

print(f'Frequency resolution: {freqs_p4[1]-freqs_p4[0]:.2f} Hz')
print(f'Band powers (P4): {[f"{b[0]}={p:.1f}" for b,p in zip(BANDS, band_powers)]}')


## 5. Interactive plot

**What to look for:**

- Energy concentrates in low bands and decreases in high bands
- Log scale reveals details across all bands
- Shaded bands represent the five brain waves



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = ['blue', 'orange', 'green', 'red']

fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                    subplot_titles=('PSD - All channels (Welch, log scale)',
                                    'Band power distribution - Channel P4'))
for i, name in enumerate(ch_names):
    f, p = psd_results[name]
    mask = f <= 80
    fig.add_trace(go.Scatter(x=f[mask], y=p[mask], name=name,
                             line=dict(color=colors[i], width=1)), row=1, col=1)
for bname, fmin, fmax, bcolor in BANDS:
    fig.add_vrect(x0=fmin, x1=fmax, fillcolor=bcolor, opacity=0.08,
                  line_width=0, row=1, col=1)
fig.update_xaxes(range=[0, 80], row=1, col=1)
fig.update_yaxes(type='log', row=1, col=1)

bar_colors = [b[3] for b in BANDS]
bar_names = [b[0] for b in BANDS]
fig.add_trace(go.Bar(x=bar_names, y=band_powers, marker_color=bar_colors,
                     name='Band power'), row=2, col=1)

fig.update_layout(height=800, title_text='Power Spectral Density Analysis',
                  xaxis_title='Frequency (Hz)', yaxis_title='PSD (uV^2/Hz)',
                  xaxis2_title='Frequency band', yaxis2_title='Absolute power (uV^2)',
                  showlegend=True)
fig.show()


## What did we learn?

- PSD reveals signal energy distribution across frequencies
- Welch's method reduces variance by averaging overlapping segments
- The five brain wave bands are clearly visible in the spectrum
- Log scale is essential to reveal details in high bands

